In [ ]:
from rdkit.Chem import rdFingerprintGenerator
import bitbirch.bitbirch as bb
from rdkit import Chem
import pandas as pd
import numpy as np
from tqdm import tqdm
import h5py
import os

In [ ]:
# Load smiles
root = "."
df = pd.read_csv(os.path.join(root, "..", "processed", "enamine_REAL_characterization", "enamine_REAL.tsv"), sep='\t')[:100000]
df['index'] = df.index

# Build the generator once
morganGen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

In [ ]:
# Given a SMILES, return its fingerprint
def morgan(smiles):
    m = Chem.MolFromSmiles(smiles)
    if not m: return None
    fp = morganGen.GetFingerprint(m)
    arr = np.zeros((2048,), dtype=np.uint8)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

In [ ]:
ids = df['id'].tolist()
X = [morgan(i) for i in tqdm(df['smiles'])]
X = np.array(X, dtype=np.int64)

In [ ]:
bb.set_merge('diameter')
brc = bb.BitBirch(threshold=0.6, branching_factor=50)
brc.fit(X)

In [ ]:
labels = brc.get_assignments(len(X))

In [ ]:
len(labels), len(set(labels))